# Day 3 - Lab 1: AI-Driven Backend Development

**Objective:** Generate a complete FastAPI backend application, including Pydantic and SQLAlchemy models, and then perform the critical engineering task of integrating the generated code with the live SQLite database created on Day 2.

**Estimated Time:** 135 minutes

**Introduction:**
Welcome to Day 3! With our requirements and architecture defined, it's time to write code. In this lab, you will act as a senior developer guiding an AI co-pilot. Your task is to generate the full backend API for the Onboarding Tool. This involves not just generating code, but also connecting it to the live database we created yesterday, moving from a prototype to a functional, data-driven application.

For definitions of key terms used in this lab, please refer to the [GLOSSARY.md](../../GLOSSARY.md).

## Step 1: Setup

We'll set up our environment and load the `schema.sql` artifact from Day 2. This SQL file contains the `CREATE TABLE` statements that define our database structure, which is the perfect context to provide the LLM for code generation.

**Model Selection:**
For code generation, models specifically fine-tuned for coding are ideal. `gpt-4.1`, `o3`, or `codex-mini` are excellent choices. Experiment to see which one gives you the cleanest code.

**Helper Functions Used:**
- `setup_llm_client()`: To configure the API client.
- `get_completion()`: To send prompts to the LLM.
- `load_artifact()`: To read the SQL schema.
- `save_artifact()`: To save the generated Python code.
- `clean_llm_output()`: To remove markdown fences from the generated code.

In [5]:
import sys
import os

# Add the project's root directory to the Python path to ensure 'utils' can be imported.
try:
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
except IndexError:
    project_root = os.path.abspath(os.path.join(os.getcwd()))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils import setup_llm_client, get_completion, save_artifact, load_artifact, clean_llm_output

client, model_name, api_provider = setup_llm_client(model_name="gemini-2.5-pro")

# Load the SQL schema from Day 2
sql_schema = load_artifact("artifacts/schema.sql")
if not sql_schema:
    print("Warning: Could not load schema.sql. Lab may not function correctly.")

backend = load_artifact("artifacts/app/main_in_memory.py")

2025-10-29 15:04:52,064 ag_aisoftdev.utils INFO LLM Client configured provider=google model=gemini-2.5-pro latency_ms=None artifacts_path=None


## Step 2: The Challenges

Follow the challenges below to build and connect your API.

### Challenge 1 (Foundational): Generating Code with In-Memory Logic

**Task:** Generate all the necessary Python code for a FastAPI application, but with simple in-memory data storage for now. This allows us to generate and validate the code's structure before adding database complexity.

**Instructions:**
1.  Create a detailed prompt that asks the LLM to act as a senior Python developer.
2.  Provide the `sql_schema` as context.
3.  Instruct the LLM to generate three key components:
    * **Pydantic Models:** For API data validation (request/response bodies).
    * **FastAPI Endpoints:** Full CRUD (Create, Read, Update, Delete) endpoints for the `users` table.
    * **In-Memory Database:** A simple Python list to act as a temporary, fake database.
4.  The final output should be a single Python script for a `main_in_memory.py` file.
5.  Save the generated code to `app/main_in_memory.py`.

In [2]:
# TODO: Write a prompt to generate a complete FastAPI application with in-memory data storage.
in_memory_api_prompt = f"""
You are a senior Python developer with expertise in FastAPI, Pydantic, and modern Python development practices. 

I need you to generate a complete FastAPI application for an Employee Onboarding Tool with the following database schema:

{sql_schema}

**Requirements:**

1. **Pydantic Models**: Create comprehensive Pydantic models for data validation:
   - UserCreate: For creating new users (name, email, role)
   - UserResponse: For API responses (id, name, email, role)
   - UserUpdate: For updating users (optional fields)
   - OnboardingTaskCreate: For creating tasks (title, description, due_date, user_id)
   - OnboardingTaskResponse: For task responses (id, title, description, due_date, status, user_id)
   - OnboardingTaskUpdate: For updating tasks (optional fields)

2. **In-Memory Database**: Create Python lists to simulate database tables:
   - users_db: List to store user records
   - tasks_db: List to store onboarding task records
   - Include helper functions to generate unique IDs

3. **FastAPI Endpoints**: Implement full CRUD operations for both users and tasks:

   **Users endpoints:**
   - POST /users/ - Create a new user
   - GET /users/ - Get all users
   - GET /users/{{user_id}} - Get user by ID
   - PUT /users/{{user_id}} - Update user by ID
   - DELETE /users/{{user_id}} - Delete user by ID

   **Onboarding Tasks endpoints:**
   - POST /tasks/ - Create a new task
   - GET /tasks/ - Get all tasks (with optional user_id filter)
   - GET /tasks/{{task_id}} - Get task by ID
   - PUT /tasks/{{task_id}} - Update task by ID
   - DELETE /tasks/{{task_id}} - Delete task by ID

4. **Additional Features:**
   - Proper error handling with HTTP status codes
   - Input validation using Pydantic
   - Clear, descriptive error messages
   - Type hints throughout
   - Docstrings for all functions
   - CORS middleware for frontend integration

5. **Code Structure:**
   - Clean, well-organized code
   - Follow PEP 8 style guidelines
   - Use FastAPI's automatic documentation features
   - Include proper imports and dependencies

Generate a single, complete Python file that can be run directly with `uvicorn main:app --reload`. The file should be named `main_in_memory.py` and should include all necessary imports, models, in-memory storage, and endpoints.

Make sure the code is production-ready, well-commented, and follows FastAPI best practices.
"""

print("--- Generating FastAPI app with in-memory database ---")
if sql_schema:
    generated_api_code = get_completion(in_memory_api_prompt, client, model_name, api_provider)
    cleaned_code = clean_llm_output(generated_api_code, language='python')
    print(cleaned_code)
    save_artifact(cleaned_code, "app/main_in_memory.py")
else:
    print("Skipping API generation because schema is missing.")

--- Generating FastAPI app with in-memory database ---
# main_in_memory.py
# To run this application:
# 1. Install dependencies: pip install "fastapi[all]"
# 2. Run the server: uvicorn main_in_memory:app --reload

import uvicorn
from fastapi import FastAPI, HTTPException, Query, status
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, EmailStr, Field
from typing import List, Optional, Literal, Dict
from datetime import date

# ---------------------------------------------------------------------------
# 1. Application Setup
# ---------------------------------------------------------------------------

app = FastAPI(
    title="Employee Onboarding Tool API",
    description="API for managing users and their onboarding tasks. This version uses an in-memory database.",
    version="1.0.0",
    contact={
        "name": "Senior Python Developer",
        "url": "https://github.com/your-profile",
    },
)

# CORS (Cross-Origin Resource Sharing) Middleware
# 

### Challenge 2 (Intermediate): Generating Database Models and Session Code

**Task:** Now, generate the specific SQLAlchemy code required to connect our application to the live `onboarding.db` SQLite database.

**Instructions:**
1.  Create a new prompt.
2.  Provide the `sql_schema` as context again.
3.  Instruct the LLM to generate two separate pieces of code:
    * **SQLAlchemy Models:** Python classes that map to your database tables.
    * **Database Session Management:** The boilerplate code to create a database engine, session maker, and a dependency function (`get_db`) for use in FastAPI.
4.  The output should be two distinct, well-commented Python code blocks. We will integrate these manually in the next step.

In [6]:
# TODO: Write a prompt to generate SQLAlchemy models and the database session/dependency code.
db_code_prompt = f"""
You are a senior Python developer with expertise in FastAPI, SQLAlchemy, and database design patterns.

Using the provided SQL schema and FastAPI back-end code, generate two separate, production-ready pieces of code:

**Database Schema:**
{sql_schema}

**Fast API Code**
{backend}

**Requirements:**

## 1. SQLAlchemy Models
Create comprehensive SQLAlchemy models that map to the database tables with the following specifications:

- **User Model:**
  - Map all fields: id (Integer, primary key), name (String), email (String, unique), role (String with enum constraint)
  - Include proper column definitions with nullable constraints
  - Add relationship to OnboardingTask (one-to-many)
  - Include __tablename__ and proper metadata
  - Add __repr__ method for debugging

- **OnboardingTask Model:**
  - Map all fields: id (Integer, primary key), title (String), description (String, nullable), due_date (String), status (String with default), user_id (Integer, foreign key)
  - Include foreign key relationship to User model
  - Add proper cascade options for deletion
  - Include __tablename__ and proper metadata
  - Add __repr__ method for debugging

- **Additional Requirements:**
  - Use proper SQLAlchemy data types (String, Integer, etc.)
  - Include proper constraints and indexes
  - Add type hints for all attributes
  - Use relationship() for foreign key relationships
  - Include proper imports (sqlalchemy, datetime, etc.)

## 2. Database Session Management
Create a complete database session management system with:

- **Database Configuration:**
  - SQLite database connection string pointing to "onboarding.db"
  - Engine creation with proper configuration
  - SessionLocal class using sessionmaker
  - Base class for declarative models

- **Dependency Function:**
  - get_db() function for FastAPI dependency injection
  - Proper session lifecycle management (try/finally)
  - Session closing and cleanup
  - Type hints and proper return types

- **Additional Features:**
  - Database initialization function (create_tables)
  - Proper error handling
  - Connection pooling considerations
  - Environment variable support for database URL
  - Proper imports and dependencies

**Code Structure:**
- Include comprehensive docstrings
- Add type hints throughout
- Follow PEP 8 style guidelines
- Include proper error handling
- Make code production-ready and maintainable

**Output Format:**
Provide two distinct, well-commented Python code blocks:
1. First block: Complete SQLAlchemy models with all imports
2. Second block: Database session management code with all imports

Each block should be complete and runnable independently when combined with the other block. Your output should only be the written code.
"""

print("--- Generating SQLAlchemy Models and Session Code ---")
if sql_schema:
    generated_db_code = get_completion(db_code_prompt, client, model_name, api_provider)
    print("\n--- Generated Database Code ---")
    print(generated_db_code)
else:
    print("Skipping DB code generation because schema is missing.")

--- Generating SQLAlchemy Models and Session Code ---

--- Generated Database Code ---
### 1. SQLAlchemy Models (`models.py`)

```python
"""
SQLAlchemy ORM Models for the Employee Onboarding Application.

This module defines the data models that map to the database schema, including
table structures, relationships, constraints, and data types. These models
are used by the application's business logic to interact with the database
in an object-oriented way.
"""

import datetime
from typing import List, Optional

from sqlalchemy import (
    CheckConstraint,
    Column,
    Date,
    ForeignKey,
    Integer,
    String,
)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
    """Base class for all declarative models."""
    pass


class User(Base):
    """
    Represents a user in the system.

    A user can be a New Hire, Manager, or HR personnel. Each user has a unique
    email and can be assigned multiple onboarding tasks.
 

### Challenge 3 (Advanced): Integrating Live Database Logic

**Task:** This is the most critical engineering step of the lab. You will manually integrate the generated database code into the FastAPI application, replacing the in-memory logic with live database operations.

**Instructions:**
This task represents a significant jump in complexity. Follow these steps carefully in your IDE (like VS Code):

1.  Create a new, empty file named `app/main.py`.
2.  **First, copy the Pydantic models and the `app = FastAPI()` line** from your `app/main_in_memory.py` file and paste them into `app/main.py`.
3.  **Next, paste the SQLAlchemy model classes and the `get_db` dependency function** you generated in Challenge 2 into your new `app/main.py`.
4.  **Now, let's refactor the `POST /users/` endpoint.** Copy the endpoint function from the in-memory file, but replace the in-memory logic (e.g., `db.append()`) with the correct SQLAlchemy session calls: `db.add(db_user)`, `db.commit()`, and `db.refresh(db_user)`.
5.  Repeat this refactoring process for the other endpoints (GET, PUT, DELETE), replacing list manipulations with the appropriate SQLAlchemy `db.query()` methods.

This task requires you to act as the senior developer, stitching together the AI-generated components into a functional, cohesive whole. You may need to ask the LLM follow-up questions like, "How do I write a SQLAlchemy query to find a user by ID?"

## Lab Conclusion

Congratulations! You have successfully generated and assembled a complete, database-connected backend API. You used an LLM to generate the boilerplate for both the API endpoints and the database models, and then performed the crucial engineering task of integrating them. You now have a working `main.py` file in your `app` directory that can create, read, update, and delete data in a live database. In the next lab, we will write a comprehensive test suite for this API.

> **Key Takeaway:** AI excels at generating boilerplate code (like models and endpoint structures), but the developer's critical role is in the final integration and wiring of these components into a coherent, working system.